# Despiking Eddy Covariance Data with TaylorSwift

This notebook demonstrates how to use the UKDE (Universal Kernel Density Estimation) despiking
routines in `TaylorSwift` to remove electrical spikes and outliers from high-frequency eddy
covariance time series before computing fluxes.

**Method reference:** Metzger et al. (2012), *Atmos. Meas. Tech.*, 5, 1699–1717.

**Workflow:**
1. Load a TOA5 file from a Campbell Scientific datalogger
2. Inspect the raw data and identify spikes visually
3. Despike individual columns with `ukde_despike()`
4. Despike all sensor channels at once with `despike_dataframe()`
5. Inspect the results with before/after plots

In [ ]:
# Install the package if needed (uncomment the line below)
# !pip install -e .. --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# TaylorSwift imports
from TaylorSwift.io import read_toa5
from TaylorSwift.cleaning import ukde_despike, despike_dataframe, polars_ukde_despike

print(f"polars: {pl.__version__}")
print(f"numpy:  {np.__version__}")

## 1. Load the TOA5 Data

TOA5 is the default ASCII output format from Campbell Scientific loggers (LoggerNet / EasyFlux DL).
Each file has a 4-line header followed by the data rows.

The example file `data/TOA5_ExampleSite_2024_07_15_1200.dat` contains a synthetic 30-minute
block at 20 Hz (36 000 records) with 30 artificial spikes injected into the wind and temperature
channels.

In [ ]:
DATA_FILE = Path("data/TOA5_ExampleSite_2024_07_15_1200.dat")

df_raw, metadata = read_toa5(DATA_FILE)

print(f"Records:   {len(df_raw):,}")
print(f"Columns:   {df_raw.columns}")
print(f"Time range: {df_raw['TIMESTAMP'][0]}  →  {df_raw['TIMESTAMP'][-1]}")
print(f"\nStation: {metadata['station_id']}")
print(f"Logger:  {metadata['logger_model']}")
print("\nUnits:")
for col, unit in metadata['units'].items():
    print(f"  {col:20s} {unit}")

## 2. Inspect the Raw Data

Basic statistics and a quick look at the raw time series.

In [ ]:
sensor_cols = ['Ux', 'Uy', 'Uz', 'T_SONIC', 'CO2_density', 'H2O_density']

# Summary statistics
stats = df_raw.select(sensor_cols).describe()
print(stats)

In [ ]:
# Convert timestamps to numpy datetime64 for matplotlib
timestamps = df_raw['TIMESTAMP'].to_numpy()

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Raw TOA5 Data — 30-minute block at 20 Hz', fontsize=13)

plot_pairs = [
    ('Ux', 'blue',   r'$U_x$ (m s$^{-1}$)'),
    ('Uz', 'green',  r'$U_z$ (m s$^{-1}$)'),
    ('T_SONIC', 'red', r'$T_{sonic}$ (°C)'),
]

for ax, (col, color, ylabel) in zip(axes, plot_pairs):
    values = df_raw[col].to_numpy()
    ax.plot(timestamps, values, color=color, lw=0.4, alpha=0.8, label='raw')

    # Highlight apparent spikes (> 5 σ from median as a quick heuristic)
    med, std = np.nanmedian(values), np.nanstd(values)
    spike_mask = np.abs(values - med) > 5 * std
    if spike_mask.any():
        ax.scatter(timestamps[spike_mask], values[spike_mask],
                   color='red', s=30, zorder=5, label=f'spike (>5σ): {spike_mask.sum()}')

    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
axes[-1].set_xlabel('Time (UTC)', fontsize=10)
fig.autofmt_xdate(rotation=20)
plt.tight_layout()
plt.show()

## 3. Single-Column Despiking with `ukde_despike()`

`ukde_despike()` is a pure NumPy/SciPy function that takes a 1-D array and returns a
cleaned array with spikes replaced by linearly interpolated values.

**How it works:**
- Fit a Gaussian KDE to the *bulk* of the distribution (values within 4×IQR of the median),
  so existing spikes cannot distort the bandwidth estimate.
- Flag samples whose normalised density falls below `prob_threshold × peak_density`.
- Replace flagged samples with NaN, linearly interpolate, and repeat until convergence.

**Key parameters:**
| Parameter | Default | Effect |
|-----------|---------|--------|
| `prob_threshold` | 1e-4 | Lower → fewer removals; higher → more aggressive |
| `max_iter` | 10 | Max despike passes (usually converges in 2–4) |

In [ ]:
# Extract the raw Ux array
ux_raw = df_raw['Ux'].to_numpy().astype(float)

# Run the UKDE despiking
ux_clean = ukde_despike(ux_raw, prob_threshold=1e-4, max_iter=10)

# Count modified samples
n_changed = int(np.sum(np.abs(ux_clean - ux_raw) > 1e-9))
print(f"Samples modified: {n_changed} / {len(ux_raw)}  ({100*n_changed/len(ux_raw):.3f}%)")
print(f"Raw  Ux:   mean={ux_raw.mean():.4f}  std={ux_raw.std():.4f}  "
      f"min={ux_raw.min():.2f}  max={ux_raw.max():.2f}")
print(f"Clean Ux:  mean={ux_clean.mean():.4f}  std={ux_clean.std():.4f}  "
      f"min={ux_clean.min():.2f}  max={ux_clean.max():.2f}")

In [ ]:
# Before/after comparison for Ux
# Zoom into a 60-second window around a known spike cluster
fs = 20.0  # Hz
changed_indices = np.where(np.abs(ux_clean - ux_raw) > 1e-9)[0]

if len(changed_indices) > 0:
    # Centre view on the first detected spike
    focus = changed_indices[0]
    half_win = int(30 * fs)  # ±30 s
    i0 = max(0, focus - half_win)
    i1 = min(len(ux_raw), focus + half_win)
else:
    i0, i1 = 0, int(60 * fs)

t_slice = timestamps[i0:i1]

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
fig.suptitle(r'$U_x$ — Before and After UKDE Despiking', fontsize=13)

axes[0].plot(t_slice, ux_raw[i0:i1], color='steelblue', lw=0.5, label='raw')
if len(changed_indices) > 0:
    mask = (changed_indices >= i0) & (changed_indices < i1)
    spike_idx_local = changed_indices[mask]
    axes[0].scatter(timestamps[spike_idx_local], ux_raw[spike_idx_local],
                    color='red', s=50, zorder=5, label=f'spikes detected ({mask.sum()})')
axes[0].set_ylabel(r'$U_x$ (m s$^{-1}$)')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_title('Raw', fontsize=10)

axes[1].plot(t_slice, ux_clean[i0:i1], color='darkorange', lw=0.5, label='despiked')
axes[1].set_ylabel(r'$U_x$ (m s$^{-1}$)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_title('After UKDE despiking', fontsize=10)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
axes[1].set_xlabel('Time (UTC)')

fig.autofmt_xdate(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# KDE visualisation: raw vs. cleaned distribution of Ux
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(8, 4))

for data, label, color in [
    (ux_raw,   'raw',       'steelblue'),
    (ux_clean, 'despiked',  'darkorange'),
]:
    kde = gaussian_kde(data, bw_method='silverman')
    x = np.linspace(data.min() - 0.5, data.max() + 0.5, 400)
    ax.plot(x, kde(x), color=color, lw=2, label=label)
    ax.axvline(data.mean(), color=color, lw=1, ls='--', alpha=0.6)

ax.set_xlabel(r'$U_x$ (m s$^{-1}$)', fontsize=11)
ax.set_ylabel('Probability density', fontsize=11)
ax.set_title(r'Distribution of $U_x$: raw vs. despiked', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Multi-Column Despiking with `despike_dataframe()`

For the standard eddy covariance workflow, all sensor channels need to be despiked before
computing (co)spectra.  `despike_dataframe()` wraps `ukde_despike()` and applies it to a
list of columns in one call.  It accepts both Polars and pandas DataFrames.

In [ ]:
# Despike all sensor channels
ec_columns = ['Ux', 'Uy', 'Uz', 'T_SONIC', 'CO2_density', 'H2O_density']

df_clean = despike_dataframe(
    df_raw,
    columns=ec_columns,
    prob_threshold=1e-4,
    max_iter=10,
    verbose=True,        # prints per-column spike counts
)

print(f"\nInput type:  {type(df_raw).__name__}")
print(f"Output type: {type(df_clean).__name__}")
print(f"Shape unchanged: {df_raw.shape} → {df_clean.shape}")

In [ ]:
# Side-by-side summary statistics: raw vs. cleaned
print(f"{'Column':<14} {'mean_raw':>10} {'mean_clean':>11} {'std_raw':>10} {'std_clean':>11}")
print("-" * 58)
for col in ec_columns:
    raw_arr   = df_raw[col].to_numpy()
    clean_arr = df_clean[col].to_numpy()
    print(f"{col:<14} {np.nanmean(raw_arr):10.4f} {np.nanmean(clean_arr):11.4f} "
          f"{np.nanstd(raw_arr):10.4f} {np.nanstd(clean_arr):11.4f}")

In [ ]:
# Full 30-min overview: raw (grey) vs. despiked (coloured) for all wind components + T
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle('30-minute EC Block — Raw vs. UKDE-Despiked', fontsize=13)

plot_config = [
    ('Ux',      r'$U_x$ (m s$^{-1}$)',  'steelblue'),
    ('Uy',      r'$U_y$ (m s$^{-1}$)',  'seagreen'),
    ('Uz',      r'$U_z$ (m s$^{-1}$)',  'darkorchid'),
    ('T_SONIC', r'$T_{sonic}$ (°C)',     'firebrick'),
]

for ax, (col, ylabel, color) in zip(axes, plot_config):
    raw_vals   = df_raw[col].to_numpy()
    clean_vals = df_clean[col].to_numpy()

    ax.plot(timestamps, raw_vals,   color='lightgrey',  lw=0.3, zorder=1, label='raw')
    ax.plot(timestamps, clean_vals, color=color,         lw=0.4, zorder=2, label='despiked')

    # Mark replaced samples
    changed = np.abs(clean_vals - raw_vals) > 1e-9
    if changed.any():
        ax.scatter(timestamps[changed], raw_vals[changed],
                   color='red', s=20, zorder=5, label=f'replaced ({changed.sum()})')

    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(loc='upper right', fontsize=7, ncol=3)
    ax.grid(True, alpha=0.25)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[-1].set_xlabel('Time (UTC)', fontsize=10)
fig.autofmt_xdate(rotation=20)
plt.tight_layout()
plt.show()

## 5. Polars Workflow: `polars_ukde_despike()`

For very large datasets (many files or high sample rates), `polars_ukde_despike()` offers an
FFT-based KDE variant (O(n log n) vs. O(n²)) that runs directly on a Polars DataFrame.
It performs a single despiking pass and adds a `{col}_cleaned` column rather than replacing
the original, making it easy to compare raw and cleaned values side by side.

In [ ]:
# Apply Polars FFT-KDE despiking to Ux and Uz
df_polars = df_raw  # already a polars DataFrame from read_toa5

for col in ['Ux', 'Uz', 'T_SONIC']:
    df_polars = polars_ukde_despike(df_polars, col, prob_threshold=1e-4)

print("Columns after polars_ukde_despike:")
for c in df_polars.columns:
    print(f"  {c}")

In [ ]:
# Compare original and _cleaned columns
print(f"{'Column':<20} {'mean_raw':>10} {'mean_clean':>11} {'std_raw':>9} {'std_clean':>10}")
print("-" * 62)
for col in ['Ux', 'Uz', 'T_SONIC']:
    raw   = df_polars[col].to_numpy()
    clean = df_polars[f"{col}_cleaned"].to_numpy()
    n_diff = int(np.sum(np.abs(clean - raw) > 1e-9))
    print(f"{col:<20} {np.nanmean(raw):10.4f} {np.nanmean(clean):11.4f} "
          f"{np.nanstd(raw):9.4f} {np.nanstd(clean):10.4f}   [{n_diff} changed]")

## 6. Effect of `prob_threshold` on Aggressiveness

The `prob_threshold` parameter controls how aggressively spikes are removed.  A larger value
means the algorithm will reject more samples; a smaller value is more permissive.

In [ ]:
thresholds = [1e-5, 1e-4, 1e-3, 1e-2]
ux = df_raw['Ux'].to_numpy().astype(float)

print(f"{'threshold':<12} {'n_removed':>10} {'mean':>10} {'std':>10}")
print("-" * 45)
results = {}
for thr in thresholds:
    cleaned = ukde_despike(ux, prob_threshold=thr)
    n_removed = int(np.sum(np.abs(cleaned - ux) > 1e-9))
    results[thr] = cleaned
    print(f"{thr:<12.0e} {n_removed:>10}  {cleaned.mean():>10.4f}  {cleaned.std():>10.4f}")

In [ ]:
# Zoom plot showing effect of different thresholds around a spike
changed_any = np.zeros(len(ux), dtype=bool)
for cleaned in results.values():
    changed_any |= np.abs(cleaned - ux) > 1e-9

spike_locations = np.where(changed_any)[0]
if len(spike_locations) > 0:
    focus = spike_locations[0]
    i0 = max(0, focus - int(5 * fs))
    i1 = min(len(ux), focus + int(5 * fs))

    fig, ax = plt.subplots(figsize=(10, 4))
    t_zoom = timestamps[i0:i1]
    ax.plot(t_zoom, ux[i0:i1], color='lightgrey', lw=1.5, zorder=1, label='raw')

    colors = ['steelblue', 'darkorange', 'seagreen', 'firebrick']
    for (thr, cleaned), color in zip(results.items(), colors):
        n_rem = int(np.sum(np.abs(cleaned - ux) > 1e-9))
        ax.plot(t_zoom, cleaned[i0:i1], lw=1.2, color=color,
                label=f'threshold={thr:.0e}  ({n_rem} removed)')

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax.set_xlabel('Time (UTC)')
    ax.set_ylabel(r'$U_x$ (m s$^{-1}$)')
    ax.set_title(r'Effect of `prob_threshold` on $U_x$ despiking')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("No spikes detected in this slice — try a different column or threshold.")

## 7. Save Despiked Data to a New TOA5-compatible CSV

After despiking, you may want to write the cleaned DataFrame back to disk before passing it
to `process_file()` for spectral computation.  Polars makes this straightforward.

In [ ]:
# Write the despiked DataFrame as a plain CSV (no TOA5 header)
out_path = Path("data/TOA5_ExampleSite_2024_07_15_1200_despiked.csv")
df_clean.write_csv(out_path)
print(f"Saved despiked data to {out_path}")
print(f"File size: {out_path.stat().st_size / 1e6:.2f} MB")

## Summary

| Function | Input | Output | Use case |
|----------|-------|--------|----------|
| `ukde_despike(arr)` | NumPy array | NumPy array | Single column, iterative, ≤10 000 samples |
| `despike_dataframe(df, cols)` | pandas or polars DataFrame | same type | Multiple columns, iterative — recommended for standard EC workflow |
| `polars_ukde_despike(df, col)` | polars DataFrame | polars DataFrame + `_cleaned` col | Single-pass FFT-KDE, large datasets |

**Recommended defaults for eddy covariance:**
- `prob_threshold = 1e-4` — removes clear outliers without over-despiking
- `max_iter = 10` — usually converges in 2–4 passes
- Columns to despike: `['Ux', 'Uy', 'Uz', 'T_SONIC', 'CO2_density', 'H2O_density']`

**Reference:** Metzger, S. et al. (2012), *Atmos. Meas. Tech.*, 5, 1699–1717.